In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor

: 

### Creazione e addestramento dei modelli

In [ ]:
def load_and_split_data(train_path, test_path, target_col='target_assignments'):
    """
    Carica i file CSV (già pre-processati e numerici) e separa X e y.
    """
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    # Separazione X e y
    X_train = df_train.drop(columns=[target_col])
    y_train = df_train[target_col]
    
    X_test = df_test.drop(columns=[target_col])
    y_test = df_test[target_col]
    
    print(f"Dataset caricati correttamente:")
    print(f"- Train set: {X_train.shape[0]} righe, {X_train.shape[1]} feature")
    print(f"- Test set:  {X_test.shape[0]} righe, {X_test.shape[1]} feature")
    
    return X_train, y_train, X_test, y_test

In [ ]:
def prepare_asp_facts(X_test, y_test, q10, q50, q90):
    """
    Combina le predizioni con i dati di contesto e calcola lo score di incertezza.
    """
    results_df = X_test.copy()
    
    # La predizione da suggerire all'ASP arrotondata all'intero più vicino
    results_df['predicted_assignments'] = np.round(q50).astype(int)
    
    # Valore reale (per eventuale validazione, non da passare all'ASP)
    results_df['actual_assignments'] = y_test.values
    
    # Calcolo dell'Incertezza (Prediction Interval Width)
    # Più il valore è alto, maggiore è l'incertezza del modello.
    results_df['uncertainty_score'] = q90 - q10
    
    # Estrazione del burdenScore (che è già presente in X_test in base alla tua EDA)
    # Assicuriamoci che si chiami correttamente
    burden_col = [col for col in X_test.columns if 'burdenScore' in col][0]
    
    print("\nAnteprima dei dati pronti per essere convertiti in fatti ASP:")
    display_cols = ['predicted_assignments', 'uncertainty_score', burden_col, 'density_ratio']
    display(results_df[display_cols].head())
    
    return results_df

### XGBoost

In [ ]:
def train_xgboost_quantiles(X_train, y_train, X_test):
    """
    Addestra 3 modelli XGBoost per stimare il 10°, 50° (mediana) e 90° percentile.
    Richiede xgboost >= 2.0.0.
    """
    print("Addestramento modello limite inferiore (10° percentile)...")
    xgb_q10 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.10,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q10.fit(X_train, y_train)

    print("Addestramento modello mediano (50° percentile - Predizione principale)...")
    xgb_q50 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.50,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q50.fit(X_train, y_train)

    print("Addestramento modello limite superiore (90° percentile)...")
    xgb_q90 = xgb.XGBRegressor(
        objective='reg:quantileerror', 
        quantile_alpha=0.90,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    )
    xgb_q90.fit(X_train, y_train)
    
    # Generazione delle predizioni sul Test Set
    pred_q10 = xgb_q10.predict(X_test)
    pred_q50 = xgb_q50.predict(X_test)
    pred_q90 = xgb_q90.predict(X_test)
    
    return pred_q10, pred_q50, pred_q90

In [ ]:
def train_lightgbm_quantiles(X_train, y_train, X_test):
    print("Addestramento LightGBM (10°, 50°, 90° percentile)...")
    
    # q10 - Limite inferiore
    lgb_q10 = LGBMRegressor(
        objective='quantile', 
        alpha=0.10, 
        n_estimators=100, 
        random_state=42, 
        verbose=-1
    )
    lgb_q10.fit(X_train, y_train)
    
    # q50 - Mediana (Predizione principale)
    lgb_q50 = LGBMRegressor(
        objective='quantile', 
        alpha=0.50, 
        n_estimators=100, 
        random_state=42, 
        verbose=-1
    )
    lgb_q50.fit(X_train, y_train)
    
    # q90 - Limite superiore
    lgb_q90 = LGBMRegressor(
        objective='quantile', 
        alpha=0.90, 
        n_estimators=100, 
        random_state=42, 
        verbose=-1
    )
    lgb_q90.fit(X_train, y_train)
    
    return lgb_q10.predict(X_test), lgb_q50.predict(X_test), lgb_q90.predict(X_test)

In [ ]:

def train_sklearn_gb_quantiles(X_train, y_train, X_test):
    print("Addestramento Sklearn Gradient Boosting (10°, 50°, 90° percentile)...")
    
    # q10
    gb_q10 = GradientBoostingRegressor(
        loss='quantile', 
        alpha=0.10, 
        n_estimators=100, 
        random_state=42
    )
    gb_q10.fit(X_train, y_train)
    
    # q50
    gb_q50 = GradientBoostingRegressor(
        loss='quantile', 
        alpha=0.50, 
        n_estimators=100, 
        random_state=42
    )
    gb_q50.fit(X_train, y_train)
    
    # q90
    gb_q90 = GradientBoostingRegressor(
        loss='quantile', 
        alpha=0.90, 
        n_estimators=100, 
        random_state=42
    )
    gb_q90.fit(X_train, y_train)
    
    return gb_q10.predict(X_test), gb_q50.predict(X_test), gb_q90.predict(X_test)

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def train_random_forest_quantiles(X_train, y_train, X_test):

    print("Addestramento Random Forest...")

    rf = RandomForestRegressor(
        n_estimators=200, 
        max_depth=5,            # Riduci la profondità: alberi più semplici = più divergenza
        min_samples_split=10,   # Aumenta i campioni minimi per split
        max_features='sqrt',    # Fondamentale: usa solo un sottoinsieme di feature per albero
        n_jobs=-1, 
        random_state=42
    )

    rf.fit(X_train, y_train)

    X_test_np = X_test.to_numpy()

    all_trees_preds = np.array([
        tree.predict(X_test_np)
        for tree in rf.estimators_
    ])

    pred_q10 = np.percentile(all_trees_preds, 10, axis=0)
    pred_q50 = np.percentile(all_trees_preds, 50, axis=0)
    pred_q90 = np.percentile(all_trees_preds, 90, axis=0)

    return pred_q10, pred_q50, pred_q90

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_model(model_name, y_test, q50, q10, q90):
    mae = mean_absolute_error(y_test, q50)
    rmse = np.sqrt(mean_squared_error(y_test, q50))
    r2 = r2_score(y_test, q50)

    # Verifica della copertura dell'intervallo di confidenza
    coverage = np.mean((y_test >= q10) & (y_test <= q90))

    print(f"\\n--- Metriche di Performance: {model_name} ---")
    print(f"MAE:  {mae:.3f} (pazienti)")
    print(f"RMSE: {rmse:.3f}")
    print(f"R^2:  {r2:.3f}")
    print(f"Copertura Intervallo [Q10, Q90]: {coverage*100:.1f}%")

In [ ]:
# Caricamento
X_train, y_train, X_test, y_test = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')


print("\n1. XGBoost\n")
q10_xgb, q50_xgb, q90_xgb = train_xgboost_quantiles(X_train, y_train, X_test)
evaluate_model("XGBoost", y_test, q50_xgb, q10_xgb, q90_xgb)

print("\n2. LightGBM\n")
q10_lgb, q50_lgb, q90_lgb = train_lightgbm_quantiles(X_train, y_train, X_test)
evaluate_model("LightGBM", y_test, q50_lgb, q10_lgb, q90_lgb)

print("\n3. Gradient Boosting\n")
q10_gb, q50_gb, q90_gb = train_sklearn_gb_quantiles(X_train, y_train, X_test)
evaluate_model("Sklearn GB", y_test, q50_gb, q10_gb, q90_gb)
